### 코스피 200 수익률 데이터 만들기
- ts_rtn_2008_2023은 기간 내에 한번이라도 코스피200에 들어간 기업은 모든 기간에 대한 데이터가 들어가게 됨
- 이렇게 되면 2009년~2012년에 코스피 200에 있었지만 2008년에는 없었을 때에도 2008년에도 코스피 200에 있다고 처리하고 투자종목에 들어갈 수 있음
- 년도마다 코스피 200 종목에 들어가는 티커 리스트를 뽑아서 필터링이 된 데이터를 만들어야 함

## 1. 라이브러리 호출

In [2]:
import pandas as pd
import os
import warnings
from pykrx import stock
warnings.filterwarnings("ignore")

##### 데이터 불러오기

In [3]:
rtn_df = pd.read_excel("data/ts_rtn_2008_2023.xlsx",dtype={'주식코드':str})
rtn_df

,회사명_x,주식코드,연도,총자본증가율(IFRS)_x,유형자산증가율(IFRS)_x,투자부동산증가율(IFRS)_x,비유동자산증가율(IFRS)_x,유동자산증가율(IFRS)_x,재고자산증가율(IFRS)_x,자기자본증가율(IFRS)_x,...,운전자본회전률(IFRS)_y,1회전기간(IFRS)_y,총자본투자효율(IFRS)_y,설비투자효율(IFRS)_y,기계투자효율(IFRS)_y,부가가치율(IFRS)_y,노동소득분배율(IFRS)_y,자본분배율(IFRS)_y,이윤분배율(IFRS)_y,수익률
0,(주)경방,000050,2014,1.09,-3.99,0.25,0.81,5.26,-2.77,-4.96,...,0.00,0.01,7.65,36.21,292.12,37.97,22.31,77.69,14.59,-0.155172
1,(주)경방,000050,2015,-1.22,-4.16,-1.29,-1.31,0.11,8.61,3.10,...,1.51,0.02,8.57,41.80,379.17,35.08,19.66,80.34,20.01,0.013605
2,(주)경방,000050,2016,-1.79,-3.70,-1.30,-1.64,-3.85,-10.48,2.75,...,0.00,0.02,8.89,44.22,502.67,35.04,19.01,80.99,28.16,-0.248322
3,(주)경방,000050,2017,-3.07,-7.75,-1.26,-2.50,-11.26,-33.22,3.00,...,0.52,0.02,9.12,47.66,647.91,37.96,18.25,81.75,23.48,-0.107143
4,(주)경방,000050,2018,2.29,-6.58,-1.13,0.96,23.35,58.28,2.56,...,0.60,0.02,7.22,41.34,1552.83,31.75,19.89,80.11,32.96,0.315000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,(주)카카오페이,377300,2022,7.10,89.00,0.00,55.78,-1.37,0.00,10.48,...,0.01,0.19,5.88,525.26,1023.21,35.26,68.77,31.23,56.60,-0.103399
2274,케이카(주),381970,2021,60.68,383.43,0.00,71.46,51.92,35.87,61.13,...,0.00,0.03,35.60,156.43,68177.53,10.29,37.49,62.51,23.89,-0.069881
2275,케이카(주),381970,2022,-2.85,10.12,0.00,10.59,-15.16,-7.05,-10.76,...,0.00,0.03,35.13,137.06,46254.20,8.62,41.54,58.46,16.17,0.051841
2276,(주)에프앤에프,383220,2022,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.01,46.89,920.61,0.00,38.57,7.43,92.57,61.88,0.071320


##### rtn 데이터를 연도별로 나누어 데이터 프레임 생성

In [4]:
for year in range(2008, 2024):
    globals()[f"rtn_df_{year}"] = rtn_df.loc[rtn_df['연도']==year]

## 2. 연도별로 코스피200 티커 리스트가 포함되어있는 데이터 불러오기

##### 2008~2014년은 krx 정보시스템에서 파일을 구해오고 2015~2023년은 pykrx를 통하여 티커 리스트를 가져옴

In [5]:
# kospi 폴더에 있는 파일별로 데이터 프레임 만들기
def make_df_from_files(folder_path):
    df_list = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.xlsx'):
            file_path = os.path.join(folder_path, file_name)
            df = pd.read_excel(file_path, dtype={'종목코드': str})
            df_list.append(df)
    return df_list

ko = make_df_from_files('kospi')  # kospi 폴더에 있는 모든 엑셀 파일로부터 데이터 프레임 생성

year = 2008
for i in ko:
    globals()[f"kospi_{year}"] = i['종목코드'].tolist()
    year += 1

In [7]:
stock.get_index_portfolio_deposit_file("1028", "20160102",alternative=True)

KeyboardInterrupt: 

##### 연도별 코스피 200 구성 종목 코드 반환 함수

In [6]:
def get_kospi200_tickers_by_year(year: int):
    """
    주어진 연도 기준으로 KOSPI 200 구성 종목 코드를 반환합니다.
    
    Parameters:
        year (int): 연도 (예: 2015)
        
    Returns:
        List[str]: 해당 연도 KOSPI 200 종목 코드 리스트
    """
    # KRX 기준 장이 열리는 날짜 중 가장 이른 1월 날짜 설정
    start_date = f"{year}0102"

    try:
        tickers = stock.get_index_portfolio_deposit_file("1028", start_date,alternative=True)
        return tickers
    except Exception as e:
        print(f"[{year}] 종목 코드 조회 실패: {e}")
        return []

for year in range(2015,2024):
    kospi_200_tickers = get_kospi200_tickers_by_year(year)
    globals()[f"kospi_200_{year}"] = kospi_200_tickers

KeyboardInterrupt: 

##### 연도별로 코스피 200에 들어간 데이터만 골라내기

In [ ]:
for i in range(2008,2024):
    kospi_year = globals()[f"kospi_{i}"]
    flag = globals()[f'rtn_df_{i}']['주식코드'].isin(kospi_year)
    df_200 =  globals()[f'rtn_df_{i}'].loc[flag]
    globals()[f"df_200_{i}"] = df_200

In [59]:
kospi_200_2016 = stock.get_index_portfolio_deposit_file("1028", "20160430", alternative = True)

The date you entered 20160430 seems to be a holiday. PYKRX changes the date parameter to 20160429.


In [64]:
kospi_200_2021 = stock.get_index_portfolio_deposit_file("1028", "20210430", alternative = True)

In [66]:
kospi_200_2022 = stock.get_index_portfolio_deposit_file("1028", "20220430", alternative = True)

The date you entered 20220430 seems to be a holiday. PYKRX changes the date parameter to 20220429.


In [ ]:
for i in range(2015,2024):
    globals()[f"kospi_200_{i}"] = stock.get_index_portfolio_deposit_file("1028", f"{i}0430", alternative = True)

In [ ]:
get_kospi200_tickers_by_year(202)

['005930',
 '373220',
 '207940',
 '000660',
 '051910',
 '006400',
 '005380',
 '035420',
 '000270',
 '035720',
 '005490',
 '068270',
 '028260',
 '105560',
 '012330',
 '055550',
 '003670',
 '096770',
 '066570',
 '032830',
 '034730',
 '015760',
 '033780',
 '086790',
 '003550',
 '323410',
 '051900',
 '010130',
 '329180',
 '017670',
 '009150',
 '034020',
 '036570',
 '011200',
 '018260',
 '010950',
 '000810',
 '030200',
 '003490',
 '009830',
 '316140',
 '259960',
 '090430',
 '024110',
 '377300',
 '352820',
 '086280',
 '011170',
 '011070',
 '097950',
 '302440',
 '326030',
 '000060',
 '138040',
 '383220',
 '009540',
 '271560',
 '035250',
 '032640',
 '251270',
 '047810',
 '402340',
 '005830',
 '010140',
 '267250',
 '028050',
 '034220',
 '018880',
 '021240',
 '000100',
 '078930',
 '004020',
 '008560',
 '361610',
 '161390',
 '000720',
 '012450',
 '006800',
 '282330',
 '011780',
 '011790',
 '241560',
 '128940',
 '029780',
 '004990',
 '010620',
 '008770',
 '064350',
 '036460',
 '002790',
 '003410',

In [69]:
for i in range(2015,2024):
    kospi_year = globals()[f"kospi_200_{i}"]
    flag = globals()[f"rtn_df_{i}"]['주식코드'].isin(kospi_year)
    df_200 =globals()[f"rtn_df_{i}"].loc[flag]
    globals()[f"df_200_{i}"] = df_200

In [70]:
concat_df = pd.concat([df_200_2008,df_200_2009,df_200_2010,df_200_2011,df_200_2012,df_200_2013,
df_200_2014,df_200_2015,df_200_2016,df_200_2017,df_200_2018,df_200_2019,df_200_2020,df_200_2021,df_200_2022
,df_200_2023], axis=0)
concat_df

,회사명,주식코드,연도,총자본증가율(IFRS),유형자산증가율(IFRS),투자부동산증가율(IFRS),비유동자산증가율(IFRS),유동자산증가율(IFRS),재고자산증가율(IFRS),자기자본증가율(IFRS),...,운전자본회전률(IFRS),1회전기간(IFRS),총자본투자효율(IFRS),설비투자효율(IFRS),기계투자효율(IFRS),부가가치율(IFRS),노동소득분배율(IFRS),자본분배율(IFRS),이윤분배율(IFRS),수익률
0,동화약품(주),000020,2008,11.35,33.72,0.00,29.12,2.65,48.02,11.27,...,0.00,0.01,13.28,55.27,943.98,16.71,0.00,0.00,73.46,-0.850125
32,(주)삼양홀딩스,000070,2008,-2.39,-1.32,0.00,-15.71,41.23,65.63,-9.99,...,0.13,0.02,-5.89,-17.88,-71.93,-4.94,0.00,0.00,0.00,0.068063
62,(주)유한양행,000100,2008,4.98,-0.08,0.00,-2.93,19.91,19.91,14.17,...,0.34,0.01,12.63,44.02,597.43,21.64,0.00,0.00,97.47,-0.191919
94,하이트진로홀딩스(주),000140,2008,-61.53,-98.87,0.00,-53.59,-99.61,0.00,-47.38,...,0.00,0.00,4.04,305.71,0.00,97.30,0.00,0.00,245.10,-0.233025
110,(주)두산,000150,2008,26.54,-20.64,0.00,36.75,-3.21,-33.36,124.39,...,0.24,0.01,-4.29,-26.68,-158.06,-7.98,0.00,0.00,0.00,-0.042969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,디엘이앤씨(주),375500,2023,1.65,-2.02,-2.44,1.98,1.30,-20.54,2.97,...,0.05,0.10,11.85,680.62,7842.29,16.21,66.59,33.41,18.11,0.161765
4598,(주)카카오페이,377300,2023,13.47,-7.45,0.00,42.73,5.43,0.00,8.19,...,0.02,0.17,5.76,631.32,1148.25,34.08,74.29,25.71,34.91,-0.103399
4601,케이카(주),381970,2023,3.70,5.58,0.00,2.95,4.60,-5.49,-4.44,...,0.00,0.03,35.78,137.05,50080.69,9.68,42.21,57.79,14.31,0.051841
4603,(주)에프앤에프,383220,2023,32.54,82.49,0.00,34.26,29.85,9.24,38.51,...,0.00,0.01,36.91,907.64,0.00,38.39,8.78,91.22,63.87,0.071320


In [71]:
concat_df.to_excel("data/kospi200_rtn.xlsx",index=False)

In [76]:
concat_df['수익률_1'] = concat_df.groupby('주식코드')['수익률'].shift(1)
concat_df.dropna(subset=['수익률_1'], inplace=True)

In [77]:
concat_df.shape

(2561, 119)